In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
X_train_trans=pd.read_csv("/content/ML_Project_neo_EDA_train.csv")
X_test_trans=pd.read_csv("/content/ML_Project_neo_EDA_test.csv")
y_train=pd.read_csv("/content/neo_y_train.csv")
y_test=pd.read_csv("/content/neo_y_test.csv")
df=pd.read_csv("/content/df_neo.csv")

In [ ]:
X_train_trans.isnull().sum()

,0
est_diameter_min,0
est_diameter_max,0
relative_velocity,0
miss_distance,0
orbiting_body,0
sentry_object,0
absolute_magnitude,0


In [ ]:
y_train.value_counts()

,count
hazardous,
False,65596
True,7072


In [ ]:
print(df["hazardous"].value_counts())

hazardous
False    81996
True      8840
Name: count, dtype: int64


In [ ]:
## here doing ravel for performing feature selection

y_train= y_train.values.ravel()

## downsampling

In [ ]:
import numpy as np

# Separate the two classes
df_false = df[df["hazardous"] == False]
df_true = df[df["hazardous"] == True]

# Randomly select 45,000 rows from False class
np.random.seed(42)

false_indices = np.random.choice(
    df_false.index,
    size=45000,
    replace=False
)

df_false_sampled = df_false.loc[false_indices]

# Combine the sampled False rows with all True rows
df_downsampled = pd.concat(
    [df_false_sampled, df_true]
)

# Shuffle the final dataset
df_downsampled = df_downsampled.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [ ]:
print(df_downsampled["hazardous"].value_counts())

hazardous
False    45000
True      8840
Name: count, dtype: int64


In [ ]:
y_downsampled=df_downsampled["hazardous"]
X_downsampled=df_downsampled.drop(columns=["hazardous","id","name"])

In [ ]:
# splitting the data

from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X_downsampled,y_downsampled,test_size=0.2,
                                                    random_state=42,
                                                    stratify=y_downsampled)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(43072, 7)
(10768, 7)
(43072,)
(10768,)


In [ ]:
X_train.head()


,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,sentry_object,absolute_magnitude
32828,0.014607,0.032662,101034.366028,7.158780e+07,Earth,False,26.3
52167,0.020163,0.045086,39694.387278,1.374217e+07,Earth,False,25.6
25007,0.010105,0.022596,38964.477659,4.548466e+07,Earth,False,27.1
17497,0.031956,0.071456,49417.779868,4.385720e+07,Earth,False,24.6
41143,0.006089,0.013616,57644.596130,6.157815e+04,Earth,False,28.2


In [ ]:
## data preprocessing

from sklearn.preprocessing import OrdinalEncoder, StandardScaler

from sklearn.compose import ColumnTransformer
transformer=ColumnTransformer(transformers=[("t1",StandardScaler(),[0,1,2,3,6]),
      ("t2",OrdinalEncoder(),[4,5])])


X_train_trans=transformer.fit_transform(X_train)
X_test_trans=transformer.transform(X_test)


In [ ]:
X_train_trans=pd.DataFrame(X_train_trans,columns=X_train.columns)
X_test_trans=pd.DataFrame(X_test_trans,columns=X_test.columns)
X_train_trans.head()

,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,sentry_object,absolute_magnitude
0,-0.423458,-0.423458,2.008384,1.537186,1.021918,0.0,0.0
1,-0.404871,-0.404871,-0.370335,-1.052973,0.784116,0.0,0.0
2,-0.438517,-0.438517,-0.398640,0.368363,1.293692,0.0,0.0
3,-0.365419,-0.365419,0.006731,0.295490,0.444398,0.0,0.0
4,-0.451953,-0.451953,0.325761,-1.665550,1.667381,0.0,0.0


In [ ]:
X_train_trans.isnull().sum()

,0
est_diameter_min,0
est_diameter_max,0
relative_velocity,0
miss_distance,0
orbiting_body,0
sentry_object,0
absolute_magnitude,0


In [ ]:
pd.Series(y_train).value_counts()

,count
hazardous,
False,36000
True,7072


In [ ]:
X_train_trans.head()

,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,sentry_object,absolute_magnitude
0,-0.423458,-0.423458,2.008384,1.537186,1.021918,0.0,0.0
1,-0.404871,-0.404871,-0.370335,-1.052973,0.784116,0.0,0.0
2,-0.438517,-0.438517,-0.398640,0.368363,1.293692,0.0,0.0
3,-0.365419,-0.365419,0.006731,0.295490,0.444398,0.0,0.0
4,-0.451953,-0.451953,0.325761,-1.665550,1.667381,0.0,0.0


## feature engineering


 -creating new features from diameter estimates

-avg diameter=est_diameter_min+est_diameter_max/2

-diameter_range=est_diameter_max-est_diameter_min

In [ ]:
X_train_trans["avg_diameter"] = (X_train_trans["est_diameter_min"] + X_train_trans["est_diameter_max"]) / 2
X_test_trans["avg_diameter"] = (X_test_trans["est_diameter_min"] + X_test_trans["est_diameter_max"]) / 2

X_train_trans["diameter_range"] = X_train_trans["est_diameter_max"] - X_train_trans["est_diameter_min"]
X_test_trans["diameter_range"] = X_test_trans["est_diameter_max"] - X_test_trans["est_diameter_min"]



-And creating a proximity feature from miss_distance feature

-Smaller miss distance -larger proximity score
-Larger miss distance -smaller proximity score

In [ ]:
X_train_trans["proximity_score"] = 1 / (X_train_trans["miss_distance"] + 1)
X_test_trans["proximity_score"] = 1 / (X_test_trans["miss_distance"] + 1)


-craeting size_velocity feature

-It is essentially a combined size-and-speed feature.

-A larger and faster asteroid will have a larger size_velocity.

In [ ]:

X_train_trans["size_velocity"] = (
    X_train_trans["avg_diameter"] * X_train_trans["relative_velocity"]
)

X_test_trans["size_velocity"] = (
    X_test_trans["avg_diameter"] * X_test_trans["relative_velocity"]
)
display(X_train_trans.head())

,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,sentry_object,absolute_magnitude,avg_diameter,diameter_range,proximity_score,size_velocity
0,-0.423458,-0.423458,2.008384,1.537186,1.021918,0.0,0.0,-0.423458,2.567402e-11,0.394137,-0.850467
1,-0.404871,-0.404871,-0.370335,-1.052973,0.784116,0.0,0.0,-0.404871,1.240370e-10,-18.877441,0.149938
2,-0.438517,-0.438517,-0.398640,0.368363,1.293692,0.0,0.0,-0.438517,-1.680270e-10,0.730800,0.174811
3,-0.365419,-0.365419,0.006731,0.295490,0.444398,0.0,0.0,-0.365419,-5.067224e-11,0.771908,-0.002460
4,-0.451953,-0.451953,0.325761,-1.665550,1.667381,0.0,0.0,-0.451953,1.409988e-10,-1.502516,-0.147228


- creating diameter ratio feature



In [ ]:
X_train_trans["diameter_ratio"] = (
    X_train_trans["est_diameter_max"] /
    (X_train_trans["est_diameter_min"] + 1e-10)
)

X_test_trans["diameter_ratio"] = (
    X_test_trans["est_diameter_max"] /
    (X_test_trans["est_diameter_min"] + 1e-10)
)

- creating size_miss_distance feature
- avg_diameter → how large the asteroid is
- miss_distance → how close it comes to Earth
- Large asteroid + small miss distance → higher size_miss_distance

In [ ]:
X_train_trans["size_miss_distance"] = (
    X_train_trans["avg_diameter"] / (X_train_trans["miss_distance"] + 1)
)

X_test_trans["size_miss_distance"] = (
    X_test_trans["avg_diameter"] / (X_test_trans["miss_distance"] + 1)
)

display(X_train_trans.head())


,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,sentry_object,absolute_magnitude,avg_diameter,diameter_range,proximity_score,size_velocity,diameter_ratio,size_miss_distance
0,-0.423458,-0.423458,2.008384,1.537186,1.021918,0.0,0.0,-0.423458,2.567402e-11,0.394137,-0.850467,1.0,-0.166901
1,-0.404871,-0.404871,-0.370335,-1.052973,0.784116,0.0,0.0,-0.404871,1.240370e-10,-18.877441,0.149938,1.0,7.642932
2,-0.438517,-0.438517,-0.398640,0.368363,1.293692,0.0,0.0,-0.438517,-1.680270e-10,0.730800,0.174811,1.0,-0.320468
3,-0.365419,-0.365419,0.006731,0.295490,0.444398,0.0,0.0,-0.365419,-5.067224e-11,0.771908,-0.002460,1.0,-0.282070
4,-0.451953,-0.451953,0.325761,-1.665550,1.667381,0.0,0.0,-0.451953,1.409988e-10,-1.502516,-0.147228,1.0,0.679067


In [ ]:
print(X_train_trans.nunique())

est_diameter_min       1493
est_diameter_max       1493
relative_velocity     43067
miss_distance         42981
orbiting_body          1493
sentry_object             1
absolute_magnitude        1
avg_diameter           1493
diameter_range         1493
proximity_score       42981
size_velocity         43068
diameter_ratio         1493
size_miss_distance    43017
dtype: int64


- Drop sentry_object and absolute_magnitude as they are continuos values.
- beacause in feature selection it is giving warning with these values

In [ ]:
X_train_trans=X_train_trans.drop(columns=["sentry_object","absolute_magnitude"])
X_test_trans=X_test_trans.drop(columns=["sentry_object","absolute_magnitude"])
X_train_trans.head()

,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,avg_diameter,diameter_range,proximity_score,size_velocity,diameter_ratio,size_miss_distance
0,-0.423458,-0.423458,2.008384,1.537186,1.021918,-0.423458,2.567402e-11,0.394137,-0.850467,1.0,-0.166901
1,-0.404871,-0.404871,-0.370335,-1.052973,0.784116,-0.404871,1.240370e-10,-18.877441,0.149938,1.0,7.642932
2,-0.438517,-0.438517,-0.398640,0.368363,1.293692,-0.438517,-1.680270e-10,0.730800,0.174811,1.0,-0.320468
3,-0.365419,-0.365419,0.006731,0.295490,0.444398,-0.365419,-5.067224e-11,0.771908,-0.002460,1.0,-0.282070
4,-0.451953,-0.451953,0.325761,-1.665550,1.667381,-0.451953,1.409988e-10,-1.502516,-0.147228,1.0,0.679067


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90836 entries, 0 to 90835
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  90836 non-null  int64  
 1   name                90836 non-null  object 
 2   est_diameter_min    90836 non-null  float64
 3   est_diameter_max    90836 non-null  float64
 4   relative_velocity   90836 non-null  float64
 5   miss_distance       90836 non-null  float64
 6   orbiting_body       90836 non-null  object 
 7   sentry_object       90836 non-null  bool   
 8   absolute_magnitude  90836 non-null  float64
 9   hazardous           90836 non-null  bool   
dtypes: bool(2), float64(5), int64(1), object(2)
memory usage: 5.7+ MB


### Feature selection

feature selection using chi sqare test

In [ ]:
## using f_classif test

from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectKBest
select_columns=SelectKBest(f_classif, k=10)
best_features=select_columns.fit_transform(X_train_trans, y_train)

## to display best feature names

selected_features=X_train_trans.columns[select_columns.get_support()]
print(selected_features)

Index(['est_diameter_min', 'est_diameter_max', 'relative_velocity',
       'miss_distance', 'orbiting_body', 'avg_diameter', 'diameter_range',
       'size_velocity', 'diameter_ratio', 'size_miss_distance'],
      dtype='object')


feature selection using mutual informstion


In [ ]:
## mutual information for clasification

from sklearn.feature_selection import mutual_info_classif
mutual_info=mutual_info_classif(X_train_trans,y_train)
mutual_info=pd.Series(mutual_info)
mutual_info.index=X_train_trans.columns

mutual_info.sort_values(ascending=False)

,0
est_diameter_min,0.218234
est_diameter_max,0.218119
avg_diameter,0.216959
orbiting_body,0.216274
diameter_range,0.214823
size_miss_distance,0.081961
diameter_ratio,0.067965
size_velocity,0.039592
relative_velocity,0.029015
proximity_score,0.010472


feature selection using varience threshold

In [ ]:
from sklearn.feature_selection import VarianceThreshold
var_thres=VarianceThreshold(threshold=0.1)
var_thres.fit_transform(X_train_trans)
best_features=X_train_trans.columns[var_thres.get_support()]
print(best_features)

Index(['est_diameter_min', 'est_diameter_max', 'relative_velocity',
       'miss_distance', 'orbiting_body', 'avg_diameter', 'proximity_score',
       'size_velocity', 'size_miss_distance'],
      dtype='object')


drop the unnecessary columns

In [ ]:
X_train_trans=X_train_trans.drop(columns="diameter_range")
X_test_trans=X_test_trans.drop(columns="diameter_range")

print(X_train_trans.shape)
print(X_test_trans.shape)


(43072, 10)
(10768, 10)


In [ ]:
X_train_trans.head()

,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,avg_diameter,proximity_score,size_velocity,diameter_ratio,size_miss_distance
0,-0.423458,-0.423458,2.008384,1.537186,1.021918,-0.423458,0.394137,-0.850467,1.0,-0.166901
1,-0.404871,-0.404871,-0.370335,-1.052973,0.784116,-0.404871,-18.877441,0.149938,1.0,7.642932
2,-0.438517,-0.438517,-0.398640,0.368363,1.293692,-0.438517,0.730800,0.174811,1.0,-0.320468
3,-0.365419,-0.365419,0.006731,0.295490,0.444398,-0.365419,0.771908,-0.002460,1.0,-0.282070
4,-0.451953,-0.451953,0.325761,-1.665550,1.667381,-0.451953,-1.502516,-0.147228,1.0,0.679067


## Model Building KNN

KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knn_model=KNeighborsClassifier()
knn_model.fit(X_train_trans,y_train)

KNeighborsClassifier()

In [ ]:
##prediction
y_pred=knn_model.predict(X_test_trans)

## Hyperparameter Tuning

Gridsearch Cv

In [ ]:

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

In [ ]:
tuned_parameters=[{"n_neighbors":[3,5,7,9,11,15,21],
                   "p":[1,2,3]}]

clf_knn= GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=tuned_parameters,
    scoring="f1",
    cv=5,
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

clf_knn.fit(X_train_trans,y_train)

Fitting 5 folds for each of 21 candidates, totalling 105 fits


GridSearchCV(cv=5, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid=[{'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
                          'p': [1, 2, 3]}],
             return_train_score=True, scoring='f1', verbose=1)

In [ ]:
print("Best Prameters:",clf_knn.best_params_)
print("Best Estimator:",clf_knn.best_estimator_)
print()

print("Score on test data:",clf_knn.score(X_test_trans, y_test))

Best Prameters: {'n_neighbors': 3, 'p': 1}
Best Estimator: KNeighborsClassifier(n_neighbors=3, p=1)

Score on test data: 0.5208211143695015


In [ ]:
print(clf_knn.best_params_)
print(clf_knn.best_score_)

y_pred = clf_knn.predict(X_test_trans)
print(classification_report(y_test, y_pred))

{'n_neighbors': 3, 'p': 1}
0.5353657041973392
              precision    recall  f1-score   support

       False       0.90      0.92      0.91      9000
        True       0.54      0.50      0.52      1768

    accuracy                           0.85     10768
   macro avg       0.72      0.71      0.72     10768
weighted avg       0.84      0.85      0.85     10768



In [ ]:
print(X_train_trans.shape)
print(X_test_trans.shape)

(43072, 10)
(10768, 10)


## Naive Bayes Model Building

Iam Using Guassian Naive Bayes for my project

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(X_train_trans,y_train)

GaussianNB()

In [ ]:
##prediction
y_pred=nb_model.predict(X_test_trans)

In [ ]:
from sklearn.model_selection import GridSearchCV

tuned_parameters_nb = {"var_smoothing": [1e-11, 1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5]}

clf_nb = GridSearchCV(
    estimator=GaussianNB(),
    param_grid=tuned_parameters_nb,
    scoring="f1",
    cv=5,
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

clf_nb.fit(X_train_trans,y_train)

Fitting 5 folds for each of 7 candidates, totalling 35 fits


GridSearchCV(cv=5, estimator=GaussianNB(), n_jobs=-1,
             param_grid={'var_smoothing': [1e-11, 1e-10, 1e-09, 1e-08, 1e-07,
                                           1e-06, 1e-05]},
             return_train_score=True, scoring='f1', verbose=1)

In [ ]:
print("Best Prameters:",clf_nb.best_params_)
print("Best Estimator:",clf_nb.best_estimator_)
print()

print("Score on test data:",clf_nb.score(X_test_trans, y_test))

Best Prameters: {'var_smoothing': 1e-09}
Best Estimator: GaussianNB()

Score on test data: 0.5435709932967703


In [ ]:
y_pred = clf_nb.predict(X_test_trans)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.94      0.80      0.86      9000
        True       0.42      0.76      0.54      1768

    accuracy                           0.79     10768
   macro avg       0.68      0.78      0.70     10768
weighted avg       0.86      0.79      0.81     10768



### Decision Tree Model Building

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt_model=DecisionTreeClassifier()
dt_model.fit(X_train_trans,y_train)


DecisionTreeClassifier()

In [ ]:
##prediction
y_pred=dt_model.predict(X_test_trans)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

tuned_parameters_dt = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

clf_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=tuned_parameters_dt,
    scoring="f1",
    cv=3,
    return_train_score=True,
    verbose=1,
    n_jobs=1
)

clf_dt.fit(X_train_trans,y_train)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42), n_jobs=1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 5, 7, 10, 15, 20],
                         'min_samples_leaf': [1, 2, 5],
                         'min_samples_split': [2, 5, 10]},
             return_train_score=True, scoring='f1', verbose=1)

In [ ]:
print("Best Prameters:",clf_dt.best_params_)
print("Best Estimator:",clf_dt.best_estimator_)
print()

print("Score on test data:",clf_dt.score(X_test_trans, y_test))

Best Prameters: {'criterion': 'gini', 'max_depth': 20, 'min_samples_leaf': 5, 'min_samples_split': 2}
Best Estimator: DecisionTreeClassifier(max_depth=20, min_samples_leaf=5, random_state=42)

Score on test data: 0.46902065289806794


In [ ]:
y_pred = clf_dt.predict(X_test_trans)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.89      0.94      0.91      9000
        True       0.57      0.40      0.47      1768

    accuracy                           0.85     10768
   macro avg       0.73      0.67      0.69     10768
weighted avg       0.84      0.85      0.84     10768



## Logistic Regression Model Buiding

In [ ]:
from sklearn.linear_model import LogisticRegression
lr_model=LogisticRegression()
lr_model.fit(X_train_trans,y_train)


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [ ]:
##prediction
y_pred=lr_model.predict(X_test_trans)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

tuned_parameters_lr = [
    {"C": [0.01, 0.1, 1, 10, 100],
     "solver": ["liblinear"],
     "penalty": ["l1", "l2"]},
    {"C": [0.01, 0.1, 1, 10, 100],
     "solver": ["lbfgs"],
     "penalty": ["l2"]}]


clf_lr = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=tuned_parameters_lr,
    scoring="f1",
    cv=5,
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

clf_lr.fit(X_train_trans,y_train)

Fitting 5 folds for each of 15 candidates, totalling 75 fits


GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000, random_state=42),
             n_jobs=-1,
             param_grid=[{'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'],
                          'solver': ['liblinear']},
                         {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2'],
                          'solver': ['lbfgs']}],
             return_train_score=True, scoring='f1', verbose=1)

In [ ]:
print("Best Prameters:",clf_lr.best_params_)
print("Best Estimator:",clf_lr.best_estimator_)
print()

print("Score on test data:",clf_lr.score(X_test_trans, y_test))

Best Prameters: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Best Estimator: LogisticRegression(C=10, max_iter=1000, penalty='l1', random_state=42,
                   solver='liblinear')

Score on test data: 0.36574236574236574


In [ ]:
y_pred = clf_lr.predict(X_test_trans)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.87      0.94      0.90      9000
        True       0.48      0.29      0.37      1768

    accuracy                           0.83     10768
   macro avg       0.68      0.62      0.63     10768
weighted avg       0.81      0.83      0.82     10768



## SVC model building

In [ ]:
from sklearn.svm import SVC
lr_model=SVC()
lr_model.fit(X_train_trans,y_train)

SVC()

In [ ]:
lr_model

SVC()

In [ ]:
#from sklearn.model_selection import GridSearchCV

#param_grid = {'C': [0.1, 1, 10, 100, 1000],
			#'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
			#'kernel': ['rbf']}

#grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3,scoring="f1")

#grid.fit(X_train_trans, y_train)

In [ ]:
# from sklearn.svm import SVC
# from sklearn.model_selection import GridSearchCV


# # tuned_parameters_svc = {
# #     "C":[1],
# #     "kernel": ["linear", "rbf"],
# #     "gamma": ["scale"]
# # }
# tuned_parameters_svc = [
#    {'C': [0.1, 1, 10, 100, 1000],
# 			'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
# 			'kernel': ['rbf']}
#  ]

# clf_svc = GridSearchCV(SVC(),
#                        tuned_parameters_svc,
#                        refit = True,
#                        verbose = 3,
#                        scoring="f1")

# clf_svc.fit(X_train_trans, y_train)

In [ ]:
# print("Best Prameters:",clf_svc.best_params_)
# print("Best Estimator:",clf_svc.best_estimator_)
# print()

# print("Score on test data:",clf_svc.score(X_test_trans, y_test))

In [ ]:
# param_grid = [
#   {'C': [1, 10, 100, 1000], 'kernel': ['linear']},
#   {'C': [1, 10, 100, 1000], 'gamma': [0.001, 0.0001], 'kernel': ['rbf']},
#  ]

## bagging ensmbles

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()
rf.fit(X_train_trans, y_train)

RandomForestClassifier()

In [ ]:
##prediction
y_pred=rf.predict(X_test_trans)

In [ ]:
# with downsampling data
from sklearn.metrics import accuracy_score, f1_score, classification_report
f1 = f1_score(y_test, y_pred)

print("F1 Score:", f1)
print(classification_report(y_test, y_pred))

F1 Score: 0.5733093955715141
              precision    recall  f1-score   support

       False       0.91      0.93      0.92      9000
        True       0.61      0.54      0.57      1768

    accuracy                           0.87     10768
   macro avg       0.76      0.74      0.75     10768
weighted avg       0.86      0.87      0.86     10768



In [ ]:
from sklearn.model_selection import GridSearchCV
tuned_parameters_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2"]
}
clf_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=tuned_parameters_rf,
    scoring="f1",
    cv=3,
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

clf_rf.fit(X_train_trans,y_train)

Fitting 3 folds for each of 162 candidates, totalling 486 fits


GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 5],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200]},
             return_train_score=True, scoring='f1', verbose=1)

In [ ]:
print("Best Prameters:",clf_rf.best_params_)
print("Best Estimator:",clf_rf.best_estimator_)
print()

print("Score on test data:",clf_rf.score(X_test_trans, y_test))

Best Prameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Best Estimator: RandomForestClassifier(min_samples_split=5, n_estimators=200, random_state=42)

Score on test data: 0.5684910795282734


In [ ]:
y_pred = clf_rf.predict(X_test_trans)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.91      0.93      0.92      9000
        True       0.61      0.53      0.57      1768

    accuracy                           0.87     10768
   macro avg       0.76      0.73      0.75     10768
weighted avg       0.86      0.87      0.86     10768



## Boosting Encembles
## AdaBoosting

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

tuned_parameters_ada = {
    "n_estimators": [50],
    "learning_rate": [1.0]
}

ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(
        max_depth=1,
        random_state=42
    ),
    random_state=42
)

ada_grid = GridSearchCV(
    estimator=ada_model,
    param_grid=tuned_parameters_ada,
    scoring="f1",
    cv=3,
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

ada_grid.fit(X_train_trans, y_train)


Fitting 3 folds for each of 1 candidates, totalling 3 fits


GridSearchCV(cv=3,
             estimator=AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,
                                                                           random_state=42),
                                          random_state=42),
             n_jobs=-1,
             param_grid={'learning_rate': [1.0], 'n_estimators': [50]},
             return_train_score=True, scoring='f1', verbose=1)

In [ ]:

print("Best Parameter:",ada_grid.best_params_)
print("Best Estimator:",ada_grid.best_estimator_)
print()

print("Score on test data:",ada_grid.score(X_test_trans, y_test))

Best Parameter: {'learning_rate': 1.0, 'n_estimators': 50}
Best Estimator: AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,
                                                    random_state=42),
                   random_state=42)

Score on test data: 0.3836798744605728


## Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

tuned_parameters_gb = {
    "n_estimators": [100],
    "learning_rate": [0.1],
    "max_depth": [3],}

gb_clf = GridSearchCV(
    estimator=GradientBoostingClassifier(
    random_state=42),
    param_grid=tuned_parameters_gb,
    scoring="f1",
    cv=3,
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

gb_clf.fit(X_train_trans, y_train)


Fitting 3 folds for each of 1 candidates, totalling 3 fits


GridSearchCV(cv=3, estimator=GradientBoostingClassifier(random_state=42),
             n_jobs=-1,
             param_grid={'learning_rate': [0.1], 'max_depth': [3],
                         'n_estimators': [100]},
             return_train_score=True, scoring='f1', verbose=1)

In [ ]:
print("Best Prameters:",gb_clf.best_params_)
print("Best Estimator:",gb_clf.best_estimator_)
print()

print("Score on test data:",gb_clf.score(X_test_trans, y_test))

Best Prameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best Estimator: GradientBoostingClassifier(random_state=42)

Score on test data: 0.41451990632318503


## XG Boost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb_model = XGBClassifier(
    objective="binary:logistic", # Changed from "multi:siftprob"
    eval_metric="logloss", # Changed from "mlogloss" as it is binary classification

)

tuned_parameters_xgb = {
    "n_estimators": [100],
    "learning_rate": [0.1],
    "max_depth": [4]
}

xgb_grid = GridSearchCV(
    estimator=xgb_model,
    param_grid=tuned_parameters_xgb,
    scoring="f1",
    cv=3,
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

xgb_grid.fit(X_train_trans, y_train)

Fitting 3 folds for each of 1 candidates, totalling 3 fits


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=True,
                                     eval_metric='logloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints...
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'learning_rate': [0.1], 'max_depth': [4],
                         'n_estimators': [100]},
             return_train_score=True, scoring='f1', verbose=1)

In [ ]:
print("Best Prameters:",xgb_grid.best_params_)
print("Best Estimator:",xgb_grid.best_estimator_)
print()

print("Score on test data:",xgb_grid.score(X_test_trans, y_test))

Best Prameters: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}
Best Estimator: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

Score on test data: 0.4088269454123113


## Evaluation Metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.8674777117384844
Precision: 0.6107862248213125
Recall   : 0.5316742081447964
F1 Score : 0.5684910795282734


In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[8401  599]
 [ 828  940]]


In [ ]:
import pickle


In [ ]:
with open("model_pipe.pkl","wb") as f:
  pickle.dump(clf_rf,f)